# Evaluación del detector de duplicados

| Bloque | Qué se mide | Con qué |
|---|---|---|
| 1 Extractor | idempotencia e integridad | 3 corridas sobre un PDF de 3 páginas + 2 jueces |
| 3 Reglas | alucinaciones, omisiones, idempotencia | golden de 32 casos |
| 3b LLM + skill | lo mismo + citas | el mismo golden, con guardrail |
| Guardrail | ¿frena referencias inventadas? | salidas corrompidas a propósito |
| 4 Cruzador | — | sin evaluación (ficticio) |
| 5 Trazador | completitud y geometría | `hallazgos_trazados.pdf` + revisión humana |

**Antes de correr:** cargá `DOCAI_ENDPOINT` y `GCP_PROJECT` en los Secretos de Colab (🔑), o como variables de entorno.

**Archivos en `/content`:**
- `detalle_(1)_ANONIMIZADO_sin_afiliado-1-3.pdf`
- los 3 CSV de la extracción del lote (`detalle_*_ANONIMIZADO_sin_afiliado.csv`)
- `golden_duplicados.csv`
- `hallazgos_trazados.pdf`
- opcional: los 3 PDFs originales, para verificar que el trazador no alteró las imágenes

In [ ]:
# --- Configuración ------------------------------------------------------------------------------
%pip install -q pymupdf google-genai

import base64, hashlib, json, os, math, random, re, time, unicodedata
from bisect import bisect_right
from collections import Counter
from decimal import Decimal, InvalidOperation
from pathlib import Path
from statistics import median
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymupdf
from IPython.display import Image, display
from matplotlib.colors import to_rgb
from matplotlib.patches import Patch
from pydantic import BaseModel

C = Path("/content")
PDF_EVAL = C / "detalle_(1)_ANONIMIZADO_sin_afiliado-1-3.pdf"
LOTE = {"L0": "detalle_ANONIMIZADO_sin_afiliado", "L1": "detalle_(1)_ANONIMIZADO_sin_afiliado",
        "L2": "detalle_(2)_ANONIMIZADO_sin_afiliado"}           # nombre corto -> archivo (sin extensión)
GOLDEN_CSV = C / "golden_duplicados.csv"
TRAZADO_PDF = C / "hallazgos_trazados.pdf"
OUT = C / "eval"
OUT.mkdir(exist_ok=True)

# Datos de GCP: Secretos de Colab (🔑) o variables de entorno. No van escritos en el código.
def secreto(nombre, defecto=""):
    try:
        from google.colab import userdata
        return userdata.get(nombre)
    except Exception:
        return os.environ.get(nombre, defecto)


ENDPOINT = secreto("DOCAI_ENDPOINT")   # https://us-documentai.googleapis.com/v1/projects/<PROYECTO>/locations/us/processors/<PROCESADOR>:process
FIELD_MASK = "text,pages.pageNumber,pages.dimension,pages.tokens"
MODEL, PROJECT_ID, LOCATION = "gemini-3.8-flash", secreto("GCP_PROJECT"), "global"
PRICE_OCR = 1.50 / 1000             # USD por página
PRICE_IN, PRICE_OUT = 0.75, 3.75    # USD por 1M tokens

UMBRAL_P = 0.05       # flujo 3: duplicado potencial si P(≥2 el mismo día) < 5 %
UMBRAL_P_LLM = 0.30   # flujo 3b: filtro laxo por código antes del LLM
RUNS, RUNS_LLM = 3, 2

In [ ]:
# --- Llamadas a las APIs y registro de costos ---------------------------------------------------
COSTOS, RESUMEN = [], []
_session = _client = None


def docai(pdf_bytes):
    """Document AI online, sin caché: cada llamada se cobra."""
    global _session
    if _session is None:
        import google.auth
        from google.auth.transport.requests import AuthorizedSession
        creds = globals().get("credentials") or google.auth.default()[0]
        if getattr(creds, "requires_scopes", False):
            creds = creds.with_scopes(["https://www.googleapis.com/auth/cloud-platform"])
        _session = AuthorizedSession(creds)
    body = {"rawDocument": {"content": base64.b64encode(pdf_bytes).decode(), "mimeType": "application/pdf"},
            "fieldMask": FIELD_MASK}
    resp = _session.post(ENDPOINT, json=body, timeout=300)
    resp.raise_for_status()
    return resp.json()["document"]


def gemini(contents, schema, system, etiqueta):
    """Gemini con salida estructurada y temperatura 0."""
    global _client
    from google import genai
    from google.genai import types
    if _client is None:
        _client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION,
                               credentials=globals().get("credentials"))
    t0 = time.perf_counter()
    resp = _client.models.generate_content(
        model=MODEL, contents=contents,
        config=types.GenerateContentConfig(system_instruction=system, response_mime_type="application/json",
                                           response_schema=schema, temperature=0))
    u = resp.usage_metadata
    tin, tout = u.prompt_token_count or 0, (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)
    COSTOS.append({"llamada": etiqueta, "latencia_s": round(time.perf_counter() - t0, 2), "tokens_in": tin,
                   "tokens_out": tout, "costo_usd": (tin * PRICE_IN + tout * PRICE_OUT) / 1e6})
    return schema.model_validate_json(resp.text)


def imagen(png):
    from google.genai import types
    return types.Part.from_bytes(data=png, mime_type="image/png")


def anotar(bloque, metrica, valor):
    RESUMEN.append({"bloque": bloque, "métrica": metrica, "valor": valor})


def ok(x):
    return "✅" if x else "❌"

## 1. Extractor
El mismo PDF de 3 páginas pasa 3 veces por Document AI + parser.
- **Idempotencia:** el código compara las tablas celda por celda y el juez decide si cada diferencia importa. Al juez se lo pone a prueba con 3 diferencias conocidas.
- **Integridad:** un segundo juez compara la imagen de cada página con las filas extraídas.

Es una demostración de que el mecanismo funciona, no de que el extractor sea robusto: 3 páginas limpias son una muestra fácil. Para que la evaluación garantice algo, hay que elegir páginas difíciles: tokens con baja confianza, filas con Der.+Hon.≠Total, columnas corridas (L0 p.23–26), renglones de ruido (L2 p.10, 13, 21) y la página del TOTAL impreso.

In [ ]:
# --- Parser de planilla (el del prototipo) --------------------------------------------------------
ROW_TOL = 0.6
HEADERS = [
    ("nro_int", "Nro. Int.", "left"), ("nro_afiliado", "Nro. de Afiliado", "left"),
    ("nro_bono", "Nro. de Bono", "left"), ("profesional", "Profesional Efector", "left"),
    ("servicio", "Servicio", "left"), ("fecha", "Fecha", "left"), ("paciente", "Apellido y Nombre", "left"),
    ("prestacion", "Prestación", "left"), ("cant", "Cant.", "right"), ("der", "Der.", "right"),
    ("hon", "Hon.", "right"), ("total", "Total", "right"),
]
FIELDS = [n for n, _, _ in HEADERS]
GREEK = str.maketrans("ΑΒΕΖΗΙΚΜΝΟΡΤΥΧοα", "ABEZHIKMNOPTYXoa")


def key(s):
    s = unicodedata.normalize("NFKD", s.translate(GREEK))
    return re.sub(r"[^a-z0-9]", "", "".join(c for c in s if not unicodedata.combining(c)).lower())


def page_tokens(full, page):
    out = []
    for t in page.get("tokens", []):
        lay = t.get("layout", {})
        if lay.get("orientation", 0) not in (0, 1, "PAGE_UP", "ORIENTATION_UNSPECIFIED"):
            continue
        v = (lay.get("boundingPoly") or {}).get("normalizedVertices") or []
        if not v:
            continue
        xs, ys = [p.get("x", 0.0) for p in v], [p.get("y", 0.0) for p in v]
        segs = (lay.get("textAnchor") or {}).get("textSegments") or []
        raw = "".join(full[int(s.get("startIndex", 0)):int(s.get("endIndex", 0))] for s in segs)
        raw = raw.replace("\n", " ").translate(GREEK)
        if t.get("detectedBreak") and not raw.endswith(" "):
            raw += " "
        out.append({"raw": raw, "key": key(raw), "conf": lay.get("confidence", 0.0),
                    "x0": min(xs), "x1": max(xs), "y0": min(ys), "y1": max(ys),
                    "xc": (min(xs) + max(xs)) / 2, "yc": (min(ys) + max(ys)) / 2})
    return out


def group_rows(toks):
    if not toks:
        return []
    h = median(t["y1"] - t["y0"] for t in toks)
    rows = []
    for t in sorted(toks, key=lambda t: t["yc"]):
        if rows and abs(t["yc"] - rows[-1]["yc"]) <= ROW_TOL * h:
            rows[-1]["toks"].append(t)
            rows[-1]["yc"] = median(x["yc"] for x in rows[-1]["toks"])
        else:
            rows.append({"yc": t["yc"], "toks": [t]})
    for r in rows:
        r["toks"].sort(key=lambda t: t["x0"])
    return rows


def match_headers(toks):
    found, i = {}, 0
    for name, label, align in HEADERS:
        target = key(label)
        for j in range(i, len(toks)):
            if not toks[j]["key"]:
                continue
            acc, k = "", j
            while k < len(toks) and len(acc) < len(target) and target.startswith(acc + toks[k]["key"]):
                acc += toks[k]["key"]
                k += 1
            if acc == target:
                span = [t for t in toks[j:k] if t["key"]]
                found[name] = {"align": align, "x0": min(t["x0"] for t in span), "x1": max(t["x1"] for t in span)}
                i = k
                break
    return found


def column_cuts(found):
    cols = sorted(found.items(), key=lambda kv: kv[1]["x0"])
    cuts = []
    for (_, a), (_, b) in zip(cols, cols[1:]):
        if a["align"] == "right" and b["align"] == "right":
            cuts.append(a["x1"])
        elif a["align"] == "right":
            cuts.append((a["x1"] + b["x0"]) / 2)
        else:
            cuts.append(b["x0"])
    return [n for n, _ in cols], cuts


def parse_page(full, page):
    rows = group_rows(page_tokens(full, page))
    header = max(rows, key=lambda r: len(match_headers(r["toks"])), default=None)
    found = match_headers(header["toks"]) if header else {}
    if len(found) < len(HEADERS) // 2:
        return []
    names, cuts = column_cuts(found)
    top = max(t["y1"] for t in header["toks"])
    out = []
    for r in rows:
        if r["yc"] <= top or re.search(r"pagina\d+de\d+", "".join(t["key"] for t in r["toks"])):
            continue
        cells = dict.fromkeys(names, "")
        for t in r["toks"]:
            cells[names[bisect_right(cuts, t["xc"])]] += t["raw"]
        cells = {n: re.sub(r"\s+", " ", v.replace("|", " ")).strip().strip(" .") for n, v in cells.items()}
        if any(cells.values()):
            cells.update(pagina=page.get("pageNumber"),
                         x0=min(t["x0"] for t in r["toks"]), x1=max(t["x1"] for t in r["toks"]),
                         y0=min(t["y0"] for t in r["toks"]), y1=max(t["y1"] for t in r["toks"]))
            out.append(cells)
    return out


def ar_decimal(s):
    s = (s or "").replace("$", "").replace(" ", "")
    try:
        return Decimal(s.replace(".", "").replace(",", ".")) if s else None
    except InvalidOperation:
        return None

In [ ]:
# --- 1a. Tres corridas del extractor ------------------------------------------------------------
raw = PDF_EVAL.read_bytes()
with pymupdf.open(PDF_EVAL) as d:
    N_PAG = len(d)

tablas, filas = [], []
for i in range(1, RUNS + 1):
    t0 = time.perf_counter()
    doc = docai(raw)
    seg = round(time.perf_counter() - t0, 2)
    t = pd.DataFrame([f for p in doc["pages"] for f in parse_page(doc["text"], p)])
    t[FIELDS] = t.reindex(columns=FIELDS).fillna("")
    tablas.append(t)
    COSTOS.append({"llamada": f"Document AI corrida {i}", "latencia_s": seg, "tokens_in": 0, "tokens_out": 0,
                   "costo_usd": N_PAG * PRICE_OCR})
    cuadra = [all(isinstance(v, Decimal) for v in x) and x[0] + x[1] == x[2]
              for x in zip(*(t[c].map(ar_decimal) for c in ("der", "hon", "total")))]
    filas.append({"corrida": i, "renglones": len(t), "por página": t.groupby("pagina").size().tolist(),
                  "sin Nro. Int.": int((t["nro_int"] == "").sum()), "Der.+Hon.≠Total": cuadra.count(False),
                  "huella de la tabla": hashlib.sha256(t[FIELDS].to_csv(index=False).encode()).hexdigest()[:12],
                  "latencia_s": seg})

identicas = len({f["huella de la tabla"] for f in filas}) == 1
display(pd.DataFrame(filas).set_index("corrida"))
print(f"{ok(identicas)} Las {RUNS} corridas dan exactamente la misma tabla")
anotar("1 Extractor", f"tablas idénticas en {RUNS} corridas", ok(identicas))

In [ ]:
# --- 1b. Juez de idempotencia --------------------------------------------------------------------
class Diferencia(BaseModel):
    id: int
    materialidad: Literal["MATERIAL", "COSMETICA"]
    motivo: str


class Veredicto(BaseModel):
    veredicto: Literal["IDEMPOTENTE", "IDEMPOTENTE_CON_DIFERENCIAS_COSMETICAS", "NO_IDEMPOTENTE"]
    diferencias: list[Diferencia]
    resumen: str


JUEZ_IDEM = """Sos juez de calidad de un extractor de planillas de facturación médica. El mismo PDF se
procesó varias veces y recibís las diferencias exactas entre corridas, ya calculadas por código.
Juzgá cada una por su id: MATERIAL si cambia un dato para detectar duplicados o calcular débitos
(Nro. Int., fecha, prestación, cantidad, importes) o falta un renglón; COSMETICA si una persona leería
lo mismo (espacios, puntuación, letras griegas iguales a latinas). Veredicto: IDEMPOTENTE sin
diferencias; IDEMPOTENTE_CON_DIFERENCIAS_COSMETICAS si todas son cosméticas; NO_IDEMPOTENTE si hay
alguna material. Respondé en español, breve."""


def diferencias(tablas):
    """Diferencias celda por celda, con los renglones identificados por (página, Nro. Int.)."""
    filas = [{f"p.{r.pagina} · {r.nro_int or i}": {c: getattr(r, c) for c in FIELDS}
              for i, r in enumerate(t.sort_values(["pagina", "y0"]).itertuples())} for t in tablas]
    diffs = []
    for k in sorted(set().union(*filas)):
        if not all(k in f for f in filas):
            diffs.append({"renglon": k, "campo": "(renglón)", "valores": ["está" if k in f else "falta" for f in filas]})
            continue
        diffs += [{"renglon": k, "campo": c, "valores": [f[k][c] for f in filas]}
                  for c in FIELDS if len({f[k][c] for f in filas}) > 1]
    return [{"id": i, **d} for i, d in enumerate(diffs, 1)]


diffs = diferencias(tablas)
v = gemini(json.dumps(diffs, ensure_ascii=False), Veredicto, JUEZ_IDEM, "juez de idempotencia")
print(f"Diferencias exactas entre corridas: {len(diffs)} · veredicto del juez: {v.veredicto}")

# Prueba del juez: 3 diferencias conocidas (2 cosméticas, 1 material) sobre una copia de la corrida 1.
control = tablas[0].copy()
fila_a = control.index[control["prestacion"].str.contains("A")][0]
otras = [i for i in control.index if i != fila_a][:2]
control.at[fila_a, "prestacion"] = control.at[fila_a, "prestacion"].replace("A", "Α", 1)   # A griega
control.at[otras[0], "servicio"] += " ."
control.at[otras[1], "total"] = f"{ar_decimal(control.at[otras[1], 'total']) + 1:.2f}".replace(".", ",")
dc = diferencias([tablas[0], control])
vc = gemini(json.dumps(dc, ensure_ascii=False), Veredicto, JUEZ_IDEM, "juez de idempotencia (prueba)")
esperado = {d["id"]: "MATERIAL" if d["campo"] == "total" else "COSMETICA" for d in dc}
bien = sum(x.materialidad == esperado.get(x.id) for x in vc.diferencias)
juez_ok = vc.veredicto == "NO_IDEMPOTENTE" and bien == len(esperado)
print(f"{ok(juez_ok)} Prueba del juez: {bien}/{len(esperado)} diferencias bien clasificadas, veredicto {vc.veredicto}")

anotar("1 Extractor", "veredicto del juez de idempotencia", v.veredicto)
anotar("1 Extractor", "juez aprueba la prueba con diferencias conocidas", ok(juez_ok))

In [ ]:
# --- 1c. Juez de integridad: la imagen contra la tabla (verde = confirmada, rojo = discrepancia) ---
CAMPOS = ["nro_int", "fecha", "prestacion", "cant", "der", "hon", "total"]


class Discrepancia(BaseModel):
    nro_int: str
    campo: str
    extraido: str
    en_imagen: str


class Integridad(BaseModel):
    filas_en_imagen: int
    discrepancias: list[Discrepancia]


JUEZ_INTEGRIDAD = """Sos auditor de una extracción OCR de una planilla de facturación médica. Recibís la
imagen de una página y las filas extraídas. Compará nro_int, fecha, prestacion, cant, der, hon y total
contra la imagen (ignorá lo tapado en negro). Contá las filas de datos de la imagen. Listá solo
discrepancias reales: un valor distinto, o una fila que falta (campo "(fila)", extraido vacío)."""

t1, discrep, total_filas = tablas[0], [], 0
with pymupdf.open(PDF_EVAL) as d:
    for n in range(1, N_PAG + 1):
        filas_pag = t1[t1["pagina"] == n]
        page = d[n - 1]
        r = gemini([imagen(page.get_pixmap(dpi=150).tobytes("png")),
                    json.dumps(filas_pag[CAMPOS].to_dict("records"), ensure_ascii=False)],
                   Integridad, JUEZ_INTEGRIDAD, f"juez de integridad p.{n}")
        mal = {x.nro_int for x in r.discrepancias}
        discrep += [{"página": n, **x.model_dump()} for x in r.discrepancias]
        total_filas += len(filas_pag)
        W, H = page.rect.width, page.rect.height
        for f in filas_pag.itertuples():
            page.draw_rect(pymupdf.Rect(f.x0 * W - 2, f.y0 * H - 2, f.x1 * W + 2, f.y1 * H + 2),
                           color=(0.8, 0, 0) if f.nro_int in mal else (0, 0.6, 0), width=1.2)
        print(f"Página {n}: {len(filas_pag)} filas extraídas · {r.filas_en_imagen} en la imagen · "
              f"{len(r.discrepancias)} discrepancias")
        display(Image(page.get_pixmap(dpi=60).tobytes("png")))

con_error = {(x["página"], x["nro_int"]) for x in discrep if x["campo"] != "(fila)"}
faltantes = sum(x["campo"] == "(fila)" for x in discrep)
print(f"{ok(not discrep)} Filas confirmadas contra la imagen: {total_filas - len(con_error)}/{total_filas} · "
      f"filas faltantes según el juez: {faltantes}")
if discrep:
    display(pd.DataFrame(discrep))
anotar("1 Extractor", "filas confirmadas contra la imagen", f"{total_filas - len(con_error)}/{total_filas}")

## 2. Golden y lote
El lote se carga desde los CSV de la extracción con la misma limpieza del prototipo: prestaciones corridas, familias y frecuencias. El golden tiene 32 casos (16 DUPLICADO / 16 NO_DUPLICADO), y cada caso es un grupo de filas.

- Un caso se cuenta como **marcado** si algún hallazgo toca sus filas.
- **Alucinación:** marcar un NO_DUPLICADO (FP).
- **Omisión:** no marcar un DUPLICADO (FN).

In [ ]:
# --- 2. Lote y golden -----------------------------------------------------------------------------
FAMILIAS = {"180161": "ECOGRAFIA TRANSVAGINAL", "861219": "ECOGRAFIA TRANSVAGINAL",
            "180106": "ECOGRAFIA MAMARIA", "861220": "ECOGRAFIA MAMARIA",
            "341018": "UROTOMOGRAFIA", "341177": "UROTOMOGRAFIA",
            "341012": "TAC OTROS ORGANOS", "3410121": "TAC OTROS ORGANOS",
            "3410123": "TAC OTROS ORGANOS", "3410126": "TAC OTROS ORGANOS"}
REPETIBLES = re.compile(r"\bCADA\b|SUBSIGUIENTE|SUBISGUIENTE|SEGUND|POR REGIO", re.I)

crudo = pd.concat([pd.read_csv(C / f"{n}.csv", dtype=str, keep_default_na=False).assign(corto=k)
                   for k, n in LOTE.items()], ignore_index=True)
PAGINAS_PDF = crudo.assign(p=crudo["pagina"].astype(int)).groupby("archivo")["p"].max().to_dict()

corrida = (crudo["prestacion"] == "") & crudo["paciente"].str.match(r"^\d{4,7}(?:X1)?\s")
crudo.loc[corrida, "prestacion"] = crudo.loc[corrida, "paciente"]       # código corrido de columna
df = crudo[crudo["nro_int"] != ""].copy()                              # fuera TOTAL y ruido
df[["codigo", "descripcion"]] = df["prestacion"].str.extract(r"^(\S+)\s*(.*)$").fillna("")
df["pagina"], df["y0"] = df["pagina"].astype(int), df["y0"].astype(float)
df["total"] = df["total"].map(lambda s: Decimal(s) if s else None)
df["dia"] = df["fecha_dt"].str[:10]
df["familia"] = df["codigo"].map(FAMILIAS).fillna(df["codigo"])
canon = df.groupby("codigo")["descripcion"].agg(lambda s: s.mode().iat[0])
df["practica"] = df["codigo"].map(FAMILIAS).fillna(df["codigo"].map(canon))
df = df.sort_values(["archivo", "pagina", "y0"]).reset_index(drop=True)
df["orden"] = df.groupby("archivo").cumcount()
df["fila_id"] = df["corto"] + "-" + df["nro_int"]
assert df["fila_id"].is_unique

dias_lote = df.loc[df["dia"] != "", "dia"].nunique()
freq = df[df["familia"] != ""].groupby("familia").agg(practica=("practica", "first"), renglones=("fila_id", "size"))
freq["p_2"] = (freq["renglones"] / dias_lote).map(lambda l: 1 - math.exp(-l) * (1 + l))
freq["repetible"] = freq.index.str.endswith("X1") | freq["practica"].str.contains(REPETIBLES)
freq["candidata"] = ~freq["repetible"] & (freq["p_2"] < UMBRAL_P)
GRUPOS = {k: g for k, g in df[(df["dia"] != "") & (df["familia"] != "")].groupby(["dia", "familia"]) if len(g) > 1}

GOLDEN = pd.read_csv(GOLDEN_CSV, dtype=str, keep_default_na=False)
GOLDEN["filas"] = [{f"{p.split()[0]}-{n}" for p, n in zip(c.paginas.split(", "), c.nro_ints.split(", "))}
                   for c in GOLDEN.itertuples()]
faltan = set().union(*GOLDEN["filas"]) - set(df["fila_id"])
assert not faltan, f"Filas del golden que no están en el lote: {faltan}"
print(f"Lote: {len(df)} filas, {dias_lote} días · golden: {GOLDEN['etiqueta'].value_counts().to_dict()}")
display(GOLDEN.groupby(["etiqueta", "estrato"], sort=False).size().to_frame("casos"))

In [ ]:
# --- Visualización: matriz de confusión + grilla caso por caso ---------------------------------------
COLOR = {"TP": "#2e7d32", "TN": "#a5d6a7", "FP": "#ef6c00", "FN": "#c62828"}
N_POS = int((GOLDEN["etiqueta"] == "DUPLICADO").sum())


def resultados(marcadas):
    """TP / FN / FP / TN por caso del golden."""
    return [("T" if bool(f & marcadas) == (e == "DUPLICADO") else "F") + ("P" if f & marcadas else "N")
            for e, f in zip(GOLDEN["etiqueta"], GOLDEN["filas"])]


def metricas(res):
    c = Counter(res)
    return {"precisión": c["TP"] / max(c["TP"] + c["FP"], 1), "recall": c["TP"] / max(c["TP"] + c["FN"], 1),
            "alucinaciones (FP)": c["FP"], "omisiones (FN)": c["FN"]}


def mostrar(sistemas, matrices):
    fig, axs = plt.subplots(1, len(matrices), figsize=(3.4 * len(matrices), 3), squeeze=False)
    for ax, nombre in zip(axs[0], matrices):
        c, m = Counter(sistemas[nombre]), metricas(sistemas[nombre])
        ax.imshow([[1, 0], [0, 1]], cmap="RdYlGn", vmin=-1, vmax=2)
        for (i, j), k in np.ndenumerate(np.array([["TP", "FN"], ["FP", "TN"]])):
            ax.text(j, i, f"{k}\n{c[k]}", ha="center", va="center", fontsize=14)
        ax.set_xticks([0, 1], ["marcado", "no marcado"])
        ax.set_yticks([0, 1], ["DUPLICADO", "NO_DUPLICADO"])
        ax.set_title(f"{nombre}\nprecisión {m['precisión']:.0%} · recall {m['recall']:.0%}", fontsize=10)
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(14, 0.5 * len(sistemas) + 1.3))
    ax.imshow([[to_rgb(COLOR[r]) for r in res] for res in sistemas.values()], aspect="auto")
    ax.set_yticks(range(len(sistemas)), list(sistemas))
    ax.set_xticks(range(len(GOLDEN)), GOLDEN["caso_id"], rotation=90, fontsize=8)
    ax.axvline(N_POS - 0.5, color="black", lw=2)
    ax.set_title("Caso por caso · izquierda: DUPLICADO · derecha: NO_DUPLICADO", fontsize=10, loc="left")
    ax.legend(handles=[Patch(color=v, label=k) for k, v in COLOR.items()], ncol=4, frameon=False,
              fontsize=8, loc="lower right", bbox_to_anchor=(1, 1))
    plt.show()

## 3. Clasificador por reglas
El golden sale de una corrida de estas mismas reglas, así que acá funciona como **test de regresión**: lo esperado es 100 %. Si alguien cambia una regla o un umbral, la grilla muestra qué casos se movieron. Para medir idempotencia, las reglas se corren 2 veces.

In [ ]:
# --- 3. Reglas contra el golden -------------------------------------------------------------------
def reglas(df, freq):
    """Flujo 3: Nro. Int. repetido + misma familia el mismo día con P(≥2) < 5 % y no repetible."""
    hs = [frozenset(g["fila_id"]) for _, g in df.groupby("nro_int") if len(g) > 1]
    return hs + [frozenset(g["fila_id"]) for (dia, fam), g in GRUPOS.items() if freq.loc[fam, "candidata"]]


t0 = time.perf_counter()
corridas = [reglas(df, freq) for _ in range(2)]
ms = (time.perf_counter() - t0) / 2 * 1000
HALLAZGOS_REGLAS = corridas[0]
idem_reglas = set(corridas[0]) == set(corridas[1])
res_reglas = [resultados(set().union(*c)) for c in corridas]
print(f"{len(HALLAZGOS_REGLAS)} hallazgos · {ms:.0f} ms por corrida · "
      f"{ok(idem_reglas)} idempotencia: las 2 corridas dan los mismos hallazgos")
mostrar({"Reglas · corrida 1": res_reglas[0], "Reglas · corrida 2": res_reglas[1]}, ["Reglas · corrida 1"])

for k, v in metricas(res_reglas[0]).items():
    anotar("3 Reglas", k, f"{v:.0%}" if isinstance(v, float) else v)
anotar("3 Reglas", "idempotencia (2 corridas)", ok(idem_reglas))

## 3b. LLM + skill
La skill describe **la misma regla del flujo 3**: el modelo tiene que llegar, explicándolo, a lo mismo que las reglas. Su salida **cita** las filas (archivo, página, Nro. Int.) de cada hallazgo, y un **guardrail** por código rechaza la decisión si cita algo que no existe o que no es de su grupo.

Pasa el golden por el flujo 3b completo: primero el filtro laxo por código (P(≥2) < 30 %) y después el LLM. Se corre 2 veces sin caché para medir su estabilidad; en producción, la idempotencia la da la caché.

In [ ]:
# --- 3b-1. Skill, salida con citas y guardrail ----------------------------------------------------
SKILL = """# Skill: duplicados potenciales (mismo criterio que el flujo de reglas)

Cada grupo es la misma práctica facturada 2 o más veces el mismo día. No hay datos del paciente.
Tu trabajo es aplicar esta regla, no tu criterio clínico.

## Regla
es_hallazgo = true solo si se cumplen las DOS condiciones:
1. La práctica NO es repetible por definición. Es repetible si:
   - algún código termina en X1 (medicamentos e insumos: se consolidan con la cantidad), o
   - la descripción dice CADA, SUBSIGUIENTE, SUBISGUIENTE, SEGUNDA/SEGUND o POR REGION
     (puede venir truncada, por ejemplo "POR REGIO").
2. Es rara: p_2_pct es menor que 5.
Si falla cualquiera de las dos, es_hallazgo = false.

## No cambian la decisión
- Que las filas estén en páginas o archivos distintos.
- Que haya 3 o más filas.
- Que los códigos sean distintos dentro del grupo (el sistema ya agrupó códigos equivalentes).
- El tipo de estudio, el servicio o tu intuición sobre si es el mismo paciente.

## Motivo
Una oración: la condición que se cumple o la que falla, con el valor de p_2_pct."""

INSTRUCCIONES = """Sos un auditor de facturación médica que aplica una regla fija descripta en la skill.
Devolvé una decisión por cada grupo_id recibido, sin omitir ni inventar grupos. Si es_hallazgo = true,
citá en `citas` todas las filas del grupo (archivo, pagina, nro_int) copiadas tal cual del input."""


class Cita(BaseModel):
    archivo: str
    pagina: int
    nro_int: str


class Decision(BaseModel):
    grupo_id: str
    es_hallazgo: bool
    motivo: str
    citas: list[Cita]


class Respuesta(BaseModel):
    decisiones: list[Decision]


def payload_grupo(gid, g):
    f = freq.loc[g["familia"].iat[0]]
    return {"grupo_id": gid, "fecha": g["dia"].iat[0], "practica": f["practica"],
            "codigos": sorted(set(g["codigo"])), "descripciones": sorted(set(g["descripcion"])),
            "veces": len(g), "p_2_pct": round(100 * f["p_2"], 2),
            "filas": [{"archivo": a, "pagina": int(p), "nro_int": n}
                      for a, p, n in zip(g["archivo"], g["pagina"], g["nro_int"])]}


# Los grupos del golden que pasan el filtro por código
ENVIAR = {}
for filas in GOLDEN["filas"]:
    g = df[df["fila_id"].isin(filas)]
    clave = (g["dia"].iat[0], g["familia"].iat[0])
    if freq.loc[clave[1], "p_2"] < UMBRAL_P_LLM:
        ENVIAR[hashlib.sha256("|".join(clave).encode()).hexdigest()[:8]] = GRUPOS[clave]
PAYLOAD = [payload_grupo(gid, g) for gid, g in ENVIAR.items()]
POR_ID = {p["grupo_id"]: p for p in PAYLOAD}
FILAS_LOTE = set(zip(df["archivo"], df["pagina"], df["nro_int"]))


def guardrail(d):
    """Motivos para rechazar una decisión (vacío = pasa): lo citado tiene que existir y ser del grupo."""
    grupo = POR_ID.get(d.grupo_id)
    if grupo is None:
        return ["grupo inexistente"]
    propias = {(f["archivo"], f["pagina"], f["nro_int"]) for f in grupo["filas"]}
    motivos = ["hallazgo sin citas"] if d.es_hallazgo and not d.citas else []
    for ct in d.citas:
        ref = (ct.archivo, ct.pagina, ct.nro_int)
        if ct.archivo not in PAGINAS_PDF:
            motivos.append("archivo inexistente")
        elif not 1 <= ct.pagina <= PAGINAS_PDF[ct.archivo]:
            motivos.append("página inexistente")
        elif ref not in FILAS_LOTE:
            motivos.append("la fila no está en esa página")
        elif ref not in propias:
            motivos.append("fila de otro grupo")
    return sorted(set(motivos))


print(f"{len(PAYLOAD)} de {len(GOLDEN)} casos del golden pasan el filtro por código y van al LLM")

In [ ]:
# --- 3b-2. LLM contra el golden -------------------------------------------------------------------
corridas_llm = []
for i in range(1, RUNS_LLM + 1):
    r = gemini(json.dumps(PAYLOAD, ensure_ascii=False), Respuesta, INSTRUCCIONES + "\n\n" + SKILL,
               f"LLM 3b corrida {i}")
    rechazos = [(d, m) for d in r.decisiones if (m := guardrail(d))]
    aceptadas = [d for d in r.decisiones if not guardrail(d)]
    marcadas = {f for d in aceptadas if d.es_hallazgo for f in ENVIAR[d.grupo_id]["fila_id"]}
    corridas_llm.append({"res": resultados(marcadas), "rechazos": rechazos,
                         "decisiones": {d.grupo_id: d for d in r.decisiones},
                         "omitidos": set(ENVIAR) - {d.grupo_id for d in r.decisiones}})

c1 = corridas_llm[0]
mostrar({"Reglas": res_reglas[0], **{f"LLM · corrida {i}": c["res"] for i, c in enumerate(corridas_llm, 1)}},
        ["Reglas", "LLM · corrida 1"])

estable = c1["res"] == corridas_llm[-1]["res"]
costo = next(x["costo_usd"] for x in COSTOS if x["llamada"] == "LLM 3b corrida 1")
print(f"{ok(not c1['rechazos'])} Guardrail: {len(c1['rechazos'])} decisiones rechazadas por citas inválidas")
print(f"{ok(not c1['omitidos'])} Grupos sin decisión: {len(c1['omitidos'])}")
print(f"{ok(estable)} Estabilidad: las {RUNS_LLM} corridas sin caché dan el mismo resultado en el golden")
print(f"Costo: USD {costo:.4f} por corrida ({len(PAYLOAD)} grupos)")
for d, m in c1["rechazos"]:
    print(f"   rechazada {d.grupo_id}: {', '.join(m)}")

# Desacuerdos con las reglas, con el motivo del modelo (para ajustar la skill)
gid_de = {frozenset(g["fila_id"]): gid for gid, g in ENVIAR.items()}
rechazados = {d.grupo_id for d, _ in c1["rechazos"]}


def motivo_llm(filas):
    gid = gid_de.get(frozenset(filas))
    if gid is None:
        return "no llegó al LLM (filtro por código)"
    d = c1["decisiones"].get(gid)
    return "sin decisión" if d is None else d.motivo + (" · rechazada por el guardrail" if gid in rechazados else "")


desac = [{"caso": c.caso_id, "golden": c.etiqueta, "práctica": c.practica, "p_2": c.p_2,
          "LLM": "marcado" if rl.endswith("P") else "no marcado", "motivo del LLM": motivo_llm(c.filas)}
         for c, rr, rl in zip(GOLDEN.itertuples(), res_reglas[0], c1["res"]) if rr != rl]
print(f"\nDesacuerdos con las reglas: {len(desac)}")
if desac:
    display(pd.DataFrame(desac).set_index("caso"))

for k, v in metricas(c1["res"]).items():
    anotar("3b LLM", k, f"{v:.0%}" if isinstance(v, float) else v)
anotar("3b LLM", "decisiones rechazadas por el guardrail", len(c1["rechazos"]))
anotar("3b LLM", "estabilidad sin caché (2 corridas)", ok(estable))
anotar("3b LLM", "costo por corrida (USD)", f"{costo:.4f}")

## Guardrail (evaluación aparte)
Por cada DUPLICADO del golden se arma una salida válida, que cita las filas reales del grupo, y se la corrompe de 7 maneras. El guardrail tiene que rechazar todas las corruptas y ninguna válida. No usa el LLM: es gratis y determinístico.

In [ ]:
# --- Guardrail: salidas corrompidas a propósito -----------------------------------------------------
rnd = random.Random(0)
validas = [Decision(grupo_id=gid, es_hallazgo=True, motivo="válida", citas=[Cita(**f) for f in POR_ID[gid]["filas"]])
           for gid, g in ENVIAR.items()
           if any(set(g["fila_id"]) == f for f in GOLDEN.loc[GOLDEN["etiqueta"] == "DUPLICADO", "filas"])]


def cita0(d, **cambios):
    return d.model_copy(update={"citas": [d.citas[0].model_copy(update=cambios)] + d.citas[1:]})


def otra_fila(d):
    propias = {(c.archivo, c.pagina, c.nro_int) for c in d.citas}
    a, p, n = rnd.choice(sorted(FILAS_LOTE - propias))
    return d.model_copy(update={"citas": d.citas + [Cita(archivo=a, pagina=p, nro_int=n)]})


CORRUPCIONES = {
    "grupo inventado": lambda d: d.model_copy(update={"grupo_id": "ffffffff"}),
    "sin citas": lambda d: d.model_copy(update={"citas": []}),
    "archivo inexistente": lambda d: cita0(d, archivo="otro_prestador.pdf"),
    "página inexistente": lambda d: cita0(d, pagina=999),
    "página equivocada": lambda d: cita0(d, pagina=d.citas[0].pagina % PAGINAS_PDF[d.citas[0].archivo] + 1),
    "Nro. Int. inventado": lambda d: cita0(d, nro_int="0000000"),
    "fila de otro grupo": otra_fila,
}

tasa = {"válida (falsas alarmas)": np.mean([bool(guardrail(d)) for d in validas])}
tasa |= {t: np.mean([bool(guardrail(f(d))) for d in validas]) for t, f in CORRUPCIONES.items()}
colores = ["#2e7d32" if (x == 0 if t.startswith("válida") else x == 1) else "#c62828" for t, x in tasa.items()]
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.barh(list(tasa), [100 * x for x in tasa.values()], color=colores)
ax.set_xlim(0, 100)
ax.set_xlabel("% de salidas rechazadas")
ax.invert_yaxis()
ax.set_title(f"Guardrail sobre {len(validas)} salidas válidas y {len(validas) * len(CORRUPCIONES)} corruptas", fontsize=10)
plt.tight_layout()
plt.show()

deteccion = float(np.mean(list(tasa.values())[1:]))
print(f"{ok(deteccion == 1)} Detección: {deteccion:.0%} · {ok(tasa['válida (falsas alarmas)'] == 0)} "
      f"falsas alarmas: {tasa['válida (falsas alarmas)']:.0%}")
anotar("Guardrail", "salidas corruptas rechazadas", f"{deteccion:.0%}")
anotar("Guardrail", "salidas válidas rechazadas", f"{tasa['válida (falsas alarmas)']:.0%}")

## 4. Cruzador
Sin evaluación, porque es ficticio. Cuando exista se evalúa como el 3b: un golden con documentación respaldatoria y un guardrail que verifique que cada cita apunte a un documento real.

## 5. Trazador
Si el PDF le sirve al auditor lo decide una persona. Lo que sí se verifica solo, comparando `hallazgos_trazados.pdf` con los hallazgos del flujo 3:
- **Completitud:** cada fila con hallazgos tiene una caja y no sobran cajas.
- **Color:** rojo si la fila es duplicado potencial, celeste si tiene redondeo (Der. + Hon. ≠ Total), naranja si tiene las dos cosas.
- **Geometría:** la caja está en la página correcta y a menos de 2 pt de la fila.
- **Imagen intacta:** las marcas son anotaciones; la imagen original no cambia (requiere los PDFs originales).

Para la parte humana hay una muestra al azar con una planilla para completar. Criterio de aceptación sugerido (regla del 3): si en *n* cajas revisadas no aparece ningún error, la tasa de error es < 3/*n* con 95 % de confianza (30 cajas → < 10 %). Como siguiente paso, un juez visual sobre los recortes, calibrado contra esa muestra, para que la persona revise solo los desacuerdos.

In [ ]:
# --- 5a. Trazador: chequeos automáticos ------------------------------------------------------------
ROJO, CELESTE, NARANJA = (1, 0, 0), (0, 0.7, 1), (1, 0.5, 0)
NOMBRE = {ROJO: "rojo", CELESTE: "celeste", NARANJA: "naranja"}


def huella_imagen(doc, i):
    return hashlib.sha256(doc.extract_image(doc[i].get_images()[0][0])["image"]).digest()


def nombre_color(c):
    return NOMBRE[min(NOMBRE, key=lambda k: sum((x - y) ** 2 for x, y in zip(k, c)))]


# Filas que el trazador tiene que marcar y con qué color: duplicado (reglas), redondeo (Der. + Hon. ≠ Total) o ambos.
dup = set().union(*HALLAZGOS_REGLAS) if HALLAZGOS_REGLAS else set()
red = {r.fila_id for r in df.itertuples()
       if r.der and r.hon and isinstance(r.total, Decimal) and Decimal(r.der) + Decimal(r.hon) != r.total}
fila = df.set_index("fila_id")
esperadas = [(fila.loc[f], "naranja" if f in dup and f in red else "rojo" if f in dup else "celeste")
             for f in sorted(dup | red)]

cajas, paginas, imagen_ok = {}, {}, []
originales = {a: C / a for a in PAGINAS_PDF if (C / a).exists()}
with pymupdf.open(TRAZADO_PDF) as out:
    for n, page in enumerate(out):
        anots = list(page.annots())
        cab = next(a.info["content"] for a in anots if a.type[1] == "FreeText" and " · pág. " in a.info["content"])
        archivo, pag = cab.split(" · pág. ")
        paginas[(archivo, int(pag))] = n
        cajas[n] = [(pymupdf.Rect(a.rect), nombre_color(a.colors["stroke"])) for a in anots if a.type[1] == "Square"]
        if archivo in originales:
            with pymupdf.open(originales[archivo]) as o:
                imagen_ok.append(huella_imagen(out, n) == huella_imagen(o, int(pag) - 1))
    alto = out[0].rect.height

usadas, chequeos = set(), []
for r, esperado in esperadas:
    n = paginas.get((r["archivo"], r["pagina"]))
    cand = sorted((abs(c.y0 + 2 - r["y0"] * alto), i) for i, (c, _) in enumerate(cajas.get(n, [])) if (n, i) not in usadas)
    hit = cand and cand[0][0] <= 2
    caja, color = cajas[n][cand[0][1]] if hit else (None, None)
    if hit:
        usadas.add((n, cand[0][1]))
    chequeos.append({"archivo": r["archivo"], "pagina": r["pagina"], "nro_int": r["nro_int"], "codigo": r["codigo"],
                     "caja": bool(hit), "desvio_pt": round(cand[0][0], 2) if hit else None,
                     "color_esperado": esperado, "color": color, "n": n, "rect": tuple(caja) if hit else None})
ch = pd.DataFrame(chequeos)
sobran = sum(len(v) for v in cajas.values()) - len(usadas)
color_ok = int((ch["color"] == ch["color_esperado"]).sum())

print(f"Filas a marcar: {ch['color_esperado'].value_counts().to_dict()}")
print(f"{ok(ch['caja'].all())} Filas con su caja: {int(ch['caja'].sum())}/{len(ch)}")
print(f"{ok(color_ok == len(ch))} Cajas con el color correcto: {color_ok}/{len(ch)}")
print(f"{ok(sobran == 0)} Cajas que no corresponden a ninguna fila: {sobran}")
print(f"{ok(ch['desvio_pt'].max() <= 2)} Desvío máximo caja-fila: {ch['desvio_pt'].max():.2f} pt")
print(f"{ok(all(imagen_ok)) if imagen_ok else '➖'} Imagen original intacta: {sum(imagen_ok)}/{len(imagen_ok)} páginas"
      + ("" if imagen_ok else " (subí los PDFs originales para verificarlo)"))
if color_ok < len(ch):
    display(ch[ch["color"] != ch["color_esperado"]].drop(columns=["n", "rect"]))
anotar("5 Trazador", "filas con su caja", f"{int(ch['caja'].sum())}/{len(ch)}")
anotar("5 Trazador", "cajas con el color correcto", f"{color_ok}/{len(ch)}")
anotar("5 Trazador", "cajas de más", sobran)
anotar("5 Trazador", "imagen original intacta", f"{sum(imagen_ok)}/{len(imagen_ok)}" if imagen_ok else "sin verificar")

In [ ]:
# --- 5b. Muestra para revisión humana -------------------------------------------------------------
MUESTRA = 6
muestra = ch[ch["caja"]].sample(min(MUESTRA, int(ch["caja"].sum())), random_state=1)
fig, axs = plt.subplots(len(muestra), 1, figsize=(12, 1.4 * len(muestra)))
with pymupdf.open(TRAZADO_PDF) as out:
    for ax, m in zip(np.atleast_1d(axs), muestra.itertuples()):
        page, rect = out[int(m.n)], pymupdf.Rect(m.rect)
        pix = page.get_pixmap(dpi=90, clip=pymupdf.Rect(0, rect.y0 - 25, page.rect.width, rect.y1 + 25))
        ax.imshow(np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width, pix.n))
        ax.set_title(f"{m.archivo} p.{m.pagina} · fila del centro: ¿Nro. Int. {m.nro_int}, código {m.codigo}, caja {m.color_esperado}?",
                     fontsize=9, loc="left")
        ax.axis("off")
plt.tight_layout()
plt.show()
muestra[["archivo", "pagina", "nro_int", "codigo", "color_esperado"]].assign(ok="", comentario="").to_csv(
    OUT / "revision_trazador.csv", index=False)
print(f"Planilla para completar a mano: {OUT / 'revision_trazador.csv'} (columna ok: sí/no)")

## Resumen

In [ ]:
# --- Resumen ---------------------------------------------------------------------------------------
display(pd.DataFrame(RESUMEN).set_index(["bloque", "métrica"]))
costos = pd.DataFrame(COSTOS).set_index("llamada")
display(costos.round(4))
print(f"Costo total de la evaluación: USD {costos['costo_usd'].sum():.4f} · {costos['latencia_s'].sum():.1f}s en llamadas")
pd.DataFrame(RESUMEN).to_csv(OUT / "resumen.csv", index=False)